In [1]:
#Installing Unsloth

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-yowagko8/unsloth_ef25ca9761bb433d907d5628b9520036
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-yowagko8/unsloth_ef25ca9761bb433d907d5628b9520036
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 22.7 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [12]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

In [13]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.


In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
)

In [17]:
data = [
    {
        "instruction": "Explain REST APIs simply.",
        "output": "REST APIs let applications communicate over HTTP using standard methods like GET, POST, PUT, and DELETE."
    },
    {
        "instruction": "What are REST APIs suitable for?",
        "output": "REST APIs are suitable for connecting web apps, mobile apps, and backend services in a simple and scalable way."
    },
    {
        "instruction": "How do REST APIs work?",
        "output": "A client sends an HTTP request to a server endpoint, and the server returns data or performs an action, often using JSON."
    },
    {
        "instruction": "What is the difference between GET and POST?",
        "output": "GET retrieves data, while POST sends data to the server to create or process something."
    },
]

from datasets import Dataset
dataset = Dataset.from_list(data)

EOS_TOKEN = tokenizer.eos_token

def format_plain(example):
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['output']}{EOS_TOKEN}"
    )
    return {"text": text}

dataset = dataset.map(format_plain)
dataset[0]["text"]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

'### Instruction:\nExplain REST APIs simply.\n\n### Response:\nREST APIs let applications communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.<|end_of_text|>'

In [18]:
dataset_text_field="text"

In [19]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
    ),
)

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

In [20]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss
1,2.811106
2,2.811107
3,2.788082
4,2.721276
5,2.603357
6,2.418206
7,2.148820
8,1.856442
9,1.552653
10,1.251162


TrainOutput(global_step=30, training_loss=1.0103396634260813, metrics={'train_runtime': 36.2954, 'train_samples_per_second': 6.612, 'train_steps_per_second': 0.827, 'total_flos': 202459680276480.0, 'train_loss': 1.0103396634260813, 'epoch': 30.0})

In [22]:
FastLanguageModel.for_inference(model)

prompt = """### Instruction:
What is the difference between REST and GraphQL?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.2,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
What is the difference between REST and GraphQL?

### Response:
REST sends simple requests to URLs, while GraphQL sends a single request that allows a client to fetch or send data in a simple way.
